<a href="https://colab.research.google.com/github/1FATIMAH1/-Data-Engineering-for-AI-Systems--DAICO/blob/main/Data_Engineering_for_AI_Systems_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — Modern Data Engineering for AI Systems

**[Fatimah ALzeer]** | SDAIA Academy - DAICO | Trainer: Mohammed Albeladi

## What this is

This notebook runs my whole capstone pipeline from start to finish in one go.

The data is the UCI Online Retail dataset, about 541,000 real sales rows from a UK online
shop. The problem with it is that it's dirty. Around a quarter of the rows have no customer
ID, cancelled orders come through as negative quantities, and some lines are priced at zero.
If you load all of that straight into a revenue table your numbers are wrong and you never
find out.

So the pipeline streams the rows through Kafka and checks every message against a Pydantic
contract. Bad rows go to a quarantine folder and a dead-letter topic with the reason they
were rejected, so nothing just disappears. What passes goes into Delta Lake in three layers
(Bronze, Silver, Gold), and before anything reaches Gold a Great Expectations check has to
pass, otherwise the pipeline stops there. There's also a RAG stage that answers questions
about the platform and cites which chunk each answer came from.

I saved this notebook with its output, because the output is what shows the pipeline
actually ran, not just that the code compiles.

## The five parts

1. **Ingestion** — real `kafka-python` producer/consumer, Pydantic data contract, quarantine + DLQ
2. **Delta Lakehouse** — Bronze / Silver / Gold, real `MERGE` upsert, schema enforcement
3. **RAG** — chunking, ChromaDB, BM25, RRF fusion, cross-encoder reranking, citations
4. **Orchestration** — the Airflow DAG that wires all of it together
5. **Quality gate + lineage** — Great Expectations checkpoint, OpenLineage START/COMPLETE/FAIL




## Step 1 — Install Dependencies

In [ ]:
!pip install pyspark==3.5.0 delta-spark==3.2.0
!pip install kafka-python "pydantic>=2.0"
!pip install chromadb sentence-transformers rank-bm25 numpy
!pip install kagglehub pandas pyarrow loguru great_expectations openlineage-python
!pip install apache-airflow


## Step 2 — Start a local Kafka broker

Colab has Java preinstalled. Give the broker ~15-20 seconds to finish starting before
running the ingestion stage.

In [ ]:
%%bash
set -e
cd /content
if ! command -v java >/dev/null 2>&1; then
  echo "Installing OpenJDK 17..."
  apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless
fi
java -version 2>&1 | head -1
if [ ! -d /content/kafka_2.13-3.7.0 ]; then
  curl -fsSL -o /content/kafka.tgz https://archive.apache.org/dist/kafka/3.7.0/kafka_2.13-3.7.0.tgz
  tar -xzf /content/kafka.tgz -C /content
fi
ls -1 /content/kafka_2.13-3.7.0/bin/kafka-server-start.sh

openjdk version "17.0.19" 2026-04-21
/content/kafka_2.13-3.7.0/bin/kafka-server-start.sh


In [ ]:
%%bash
cd /content/kafka_2.13-3.7.0

# Format the KRaft storage directory once.
if [ ! -d /tmp/kraft-combined-logs ]; then
  KAFKA_UUID=$(bin/kafka-storage.sh random-uuid)
  bin/kafka-storage.sh format -t "$KAFKA_UUID" -c config/kraft/server.properties
fi

nohup bin/kafka-server-start.sh config/kraft/server.properties > /tmp/kafka.log 2>&1 &
echo "Broker starting — poll for readiness in the next cell."


Broker starting — poll for readiness in the next cell.


## Step 3 — Get the project code on the path

This notebook imports the `src` package, so the **whole repository** has to be present, not
just the notebook file.


Running locally from `notebooks/`, neither is needed — the cell below finds the root itself.

In [ ]:
import socket, subprocess, time

def port_open(host="localhost", port=9092, timeout=1):
    with socket.socket() as s:
        s.settimeout(timeout)
        return s.connect_ex((host, port)) == 0

for i in range(90):
    if port_open():
        print(f"Kafka is listening on localhost:9092 (took {i}s).")
        break
    time.sleep(1)
else:
    print("Broker did NOT come up. Last 40 lines of /tmp/kafka.log:\n")
    print(subprocess.run(["tail", "-40", "/tmp/kafka.log"],
                         capture_output=True, text=True).stdout)
    raise SystemExit("Fix the broker before running the ingestion stage.")

print(subprocess.run(
    ["/content/kafka_2.13-3.7.0/bin/kafka-topics.sh",
     "--bootstrap-server", "localhost:9092", "--list"],
    capture_output=True, text=True).stdout or "(no topics yet — expected on a fresh broker)")


Kafka is listening on localhost:9092 (took 12s).
__consumer_offsets
retail_transactions_dlq
retail_transactions_raw



In [ ]:
import os, sys

CANDIDATES = [
    os.getcwd(),
    os.path.abspath(".."),                              # running from notebooks/
    "/content/capstone-modern-data-engineering",        # Colab, unzipped
    "/content/capstone",                                # Colab, cloned
]

REPO_ROOT = next((p for p in CANDIDATES if os.path.isdir(os.path.join(p, "src"))), None)

if REPO_ROOT is None:
    raise SystemExit(
        "Project not found. This notebook needs the whole repository, not just the .ipynb.\n"
        "In Colab, pick ONE:\n"
        "  A) upload capstone-modern-data-engineering.zip via the Files panel, then run:\n"
        "       !unzip -q -o capstone-modern-data-engineering.zip -d /content\n"
        "  B) !git clone <your-repo-url> /content/capstone\n"
        "Then re-run this cell."
    )

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("Repo root:", REPO_ROOT)
print("Package contents:", sorted(os.listdir("src")))


Repo root: /content/capstone-modern-data-engineering
Package contents: ['__init__.py', '__pycache__', 'config.py', 'ingestion', 'lakehouse', 'lineage', 'main.py', 'quality', 'rag', 'tasks.py']


## Step 4 — Kaggle credentials

The source feed is the UCI Online Retail dataset (541,909 real transactions, 2010-2011).
In Colab: add `KAGGLE_USERNAME` and `KAGGLE_KEY` as secrets (the key icon in the sidebar).

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
    print("Kaggle credentials loaded from Colab Secrets.")
except Exception:
    print("Not on Colab - export KAGGLE_USERNAME and KAGGLE_KEY in your shell instead.")


Not on Colab - export KAGGLE_USERNAME and KAGGLE_KEY in your shell instead.


## Deliverable 1 — Ingestion

Real `KafkaProducer` streams the feed into `retail_transactions_raw`. The consumer validates
every message against the Pydantic contract: accepted records go to the landing zone,
rejected records go to `quarantine_zone/` with their rejection reason **and** to the
`retail_transactions_dlq` dead-letter topic.

No anomalies are planted. The rejections below are real cancellations, null customers and
zero-priced lines that exist in the dataset.

In [ ]:
from src.tasks import task_produce, task_consume_validate

task_produce()


[LINEAGE] START    | capstone.ingestion.produce | run 019f8873-2712-7a96-8b37-457c92192669
Using Colab cache for faster access to the 'ecommerce-data' dataset.
Downloaded to: /kaggle/input/ecommerce-data
CSV files found: ['/kaggle/input/ecommerce-data/data.csv']
Using: /kaggle/input/ecommerce-data/data.csv
Shape: 541,909 rows × 8 columns
Streaming the first 5,000 rows of the feed.
Prepared 50 price corrections for existing business keys.
  [PRODUCER] sent 5,050 messages -> topic 'retail_transactions_raw'
[LINEAGE] COMPLETE | capstone.ingestion.produce | run 019f8873-2712-7a96-8b37-457c92192669


5050

In [ ]:
ingest_report = task_consume_validate()
ingest_report


[LINEAGE] START    | capstone.ingestion.consume_validate | run 019f8873-3f3a-79ee-bc01-255e19829cf0
  [CONSUMER] REJECTED @offset 5191: Value error, Quantity must be > 0 (got -1.0 — likely a cancellation)
  [CONSUMER] REJECTED @offset 5204: Value error, Quantity must be > 0 (got -1.0 — likely a cancellation)
  [CONSUMER] REJECTED @offset 5285: Value error, Quantity must be > 0 (got -12.0 — likely a cancellation)
  [CONSUMER] REJECTED @offset 5286: Value error, Quantity must be > 0 (got -24.0 — likely a cancellation)
  [CONSUMER] REJECTED @offset 5287: Value error, Quantity must be > 0 (got -24.0 — likely a cancellation)

  CONTRACT VALIDATION AT THE INGESTION BOUNDARY
  Total consumed :      5,050
  Accepted       :      3,775  (74.8%)
  Rejected       :      1,275  (25.2%)

Top rejection reasons:
rejection_reason
Value error, CustomerID is required — cannot be null     1193
Value error, Quantity must be > 0                          70
Value error, Description is required — cannot be n

{'accepted': 3775,
 'rejected': 1275,
 'quarantine_path': './quarantine_zone/contract_violations_1784700717.csv',
 'landing_path': './data/landing/accepted_records.jsonl'}

### Inspect the quarantine zone

In [ ]:
import pandas as pd

q = pd.read_csv(ingest_report["quarantine_path"])
print(f"{len(q):,} quarantined records")
q[["InvoiceNo", "StockCode", "Quantity", "UnitPrice", "CustomerID", "rejection_reason"]].head(10)


1,275 quarantined records


,InvoiceNo,StockCode,Quantity,UnitPrice,CustomerID,rejection_reason
0,C536379,D,-1,27.50,14527.0,"Value error, Quantity must be > 0 (got -1.0 — ..."
1,C536383,35004C,-1,4.65,15311.0,"Value error, Quantity must be > 0 (got -1.0 — ..."
2,C536391,22556,-12,1.65,17548.0,"Value error, Quantity must be > 0 (got -12.0 —..."
3,C536391,21984,-24,0.29,17548.0,"Value error, Quantity must be > 0 (got -24.0 —..."
4,C536391,21983,-24,0.29,17548.0,"Value error, Quantity must be > 0 (got -24.0 —..."
5,C536391,21980,-24,0.29,17548.0,"Value error, Quantity must be > 0 (got -24.0 —..."
6,C536391,21484,-12,3.45,17548.0,"Value error, Quantity must be > 0 (got -12.0 —..."
7,C536391,22557,-12,1.65,17548.0,"Value error, Quantity must be > 0 (got -12.0 —..."
8,C536391,22553,-24,1.65,17548.0,"Value error, Quantity must be > 0 (got -24.0 —..."
9,536414,22139,56,0.00,NaN,"Value error, Description is required — cannot ..."


### Confirm the dead-letter topic actually received them

In [ ]:
!kafka_2.13-3.7.0/bin/kafka-console-consumer.sh \
    --bootstrap-server localhost:9092 \
    --topic retail_transactions_dlq \
    --from-beginning --max-messages 3 --timeout-ms 10000


/bin/bash: line 1: kafka_2.13-3.7.0/bin/kafka-console-consumer.sh: No such file or directory


## Deliverable 2 — Delta Lakehouse

Bronze appends the accepted records untouched. Silver runs a real `MERGE` keyed on
`line_id`, so the price corrections update existing rows while new lines are inserted. Then
a write carrying an undeclared column is put to the table to prove schema enforcement
refuses it.

In [ ]:
from src.tasks import task_bronze

bronze_rows = task_bronze()


[LINEAGE] START    | capstone.lakehouse.bronze | run 019f8873-c9da-7c1f-8888-9226755e4080

  BRONZE — Append landing-zone records to the Delta Bronze table
Appended 3,775 records. Bronze now holds 7,550 rows.
+---------+---------+-----------------------------------+--------+---------+----------+-------+---------------+------------+--------------------------------+
|InvoiceNo|StockCode|Description                        |Quantity|UnitPrice|CustomerID|Country|InvoiceDate    |kafka_offset|ingested_at                     |
+---------+---------+-----------------------------------+--------+---------+----------+-------+---------------+------------+--------------------------------+
|536527   |22809    |SET OF 6 T-LIGHTS SANTA            |6.0     |2.95     |12662     |Germany|12/1/2010 13:04|1109        |2026-07-22T05:38:10.782889+00:00|
|536527   |84347    |ROTATING SILVER ANGELS T-LIGHT HLDR|6.0     |2.55     |12662     |Germany|12/1/2010 13:04|1110        |2026-07-22T05:38:10.782911+00:00|
|

In [ ]:
from src.tasks import task_silver

silver_metrics = task_silver()
silver_metrics


[LINEAGE] START    | capstone.lakehouse.silver | run 019f8875-0744-7771-94e6-68ad84580e22

  SILVER — MERGE (UPSERT) on business key line_id

  MERGE [base batch] — operation logged as MERGE
    rows updated  : 3,581
    rows inserted : 0

  MERGE [correction batch] — operation logged as MERGE
    rows updated  : 3,581
    rows inserted : 0

  Silver rows: 3,581 -> 3,581
+-------------+---------+---------+-----------------------------------+--------+---------+-------+----------+--------------+--------------+--------------------------------+
|line_id      |InvoiceNo|StockCode|Description                        |Quantity|UnitPrice|revenue|CustomerID|Country       |InvoiceDate   |ingested_at                     |
+-------------+---------+---------+-----------------------------------+--------+---------+-------+----------+--------------+--------------+--------------------------------+
|536365_21730 |536365   |21730    |GLASS STAR FROSTED T-LIGHT HOLDER  |6.0     |4.68     |28.08  |17850    

{'rows_before': 3581,
 'rows_after': 3581,
 'rows_inserted': 0,
 'rows_updated': 7162,
 'merge_base': {'updated': 3581, 'inserted': 0},
 'merge_corrections': {'updated': 3581, 'inserted': 0},
 'schema_enforcement_rejected_bad_write': True}

`rows_updated` above is the proof the MERGE update path fired — those are the corrected
prices matching existing business keys, not inserts.

### Delta history for the Silver table

In [ ]:
from delta.tables import DeltaTable
from src.config import SILVER_PATH
from src.lakehouse.spark_session import create_spark_session

spark = create_spark_session("Capstone_Inspect")
DeltaTable.forPath(spark, SILVER_PATH).history() \
    .select("version", "timestamp", "operation", "operationMetrics") \
    .show(truncate=False)
spark.stop()


+-------+-----------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                                                                                                                                                                                                                                        

## Deliverable 5 — Quality gate

A real Great Expectations 1.x checkpoint on Silver. It raises when it fails, and Gold sits
downstream of it, so a failed gate halts the pipeline before any aggregate is published.

In [ ]:
from src.tasks import task_quality_gate

gate_report = task_quality_gate()
gate_report


[LINEAGE] START    | capstone.quality.gate | run 019f8875-dd65-7900-b4f6-81eb528fe342


INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmp989g4jrw' for ephemeral docs site


Validating 3,581 Silver rows...


Calculating Metrics:   0%|          | 0/46 [00:00<?, ?it/s]

[GX] Real Great Expectations checkpoint success=True
  [GX] PASSED expect_column_values_to_not_be_null
  [GX] PASSED expect_column_values_to_be_unique
  [GX] PASSED expect_column_values_to_not_be_null
  [GX] PASSED expect_column_values_to_be_between
  [GX] PASSED expect_column_values_to_be_between
  [GX] PASSED expect_column_values_to_be_between
  [GX] PASSED expect_column_values_to_match_regex

Quality gate PASSED — downstream stages are allowed to run.
[LINEAGE] COMPLETE | capstone.quality.gate | run 019f8875-dd65-7900-b4f6-81eb528fe342


{'success': True, 'failed_expectations': []}

## Deliverable 2 (cont.) — Gold

A genuine aggregate: revenue by country and invoice month, not a copy of Silver.

In [ ]:
from src.tasks import task_gold

gold_rows = task_gold()


[LINEAGE] START    | capstone.lakehouse.gold | run 019f8875-fe8c-7d32-b714-45216e39612b

  GOLD — Revenue aggregate by country and invoice month
Gold table written: 7 aggregate rows (Silver held 3,581 transaction lines).
+--------------+-------------+-------------+-------------+--------------+-------------+----------+----------------+
|Country       |invoice_month|total_revenue|invoice_count|customer_count|product_count|units_sold|avg_line_revenue|
+--------------+-------------+-------------+-------------+--------------+-------------+----------+----------------+
|United Kingdom|2010-12      |86303.99     |240          |173           |1268         |50138.0   |25.12           |
|Norway        |2010-12      |1919.14      |1            |1             |73           |1852.0    |26.29           |
|France        |2010-12      |942.38       |1            |1             |20           |449.0     |47.12           |
|EIRE          |2010-12      |577.88       |3            |1             |22        

## Deliverable 3 — RAG pipeline

Chunking, ChromaDB (HNSW) dense retrieval, BM25 keyword retrieval, RRF fusion,
cross-encoder reranking, and an answer grounded in the retrieved context with citations
back to the exact chunk.

In [ ]:
from src.tasks import task_rag

task_rag()


[LINEAGE] START    | capstone.rag.pipeline | run 019f8876-6d51-7b52-97cd-53460af99157
  Capstone Stage 3 — RAG over the pipeline knowledge base

📄 12 documents → 44 chunks after splitting

📦 Building ChromaDB vector index...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

   ✅ 44 chunks indexed (HNSW backend, all-MiniLM-L6-v2 embeddings)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


QUERY: How does the ingestion stage quarantine records that break the data contract?

  🔍 Vector search:   6 candidates
  🔑 BM25 search:     6 candidates
  ⚡ RRF fusion:      6 merged candidates
  🎯 Cross-encoder reranking 6 candidates...


...

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


  📝 Top-3 chunks after reranking:
    [1] Reciprocal Rank Fusion (RRF) merges both result lists: score = Σ 1/(k + rank_i), where k=60 is the standard co...
    [2] Hybrid search combines vector semantic search with BM25 keyword search. Reciprocal Rank Fusion (RRF) merges bo...
    [3] RRF is parameter-free and consistently outperforms a weighted linear combination. Most vector databases expose...

  ⚠️  GROQ_API_KEY not set — returning the grounded context itself,
      already cited, instead of a generated answer.

  ✅ Answer:
    [Source 1] Reciprocal Rank Fusion (RRF) merges both result lists: score = Σ 1/(k + rank_i), where k=60 is the standard constant. RRF is parameter-free and consistently outperforms a weighted linear combination.
[Source 2] Hybrid search combines vector semantic search with BM25 keyword search. Reciprocal Rank Fusion (RRF) merges both result lists: score = Σ 1/(k + rank_i), where k=60 is the standard constant.
[Source 3] RRF is parameter-free and consistently

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


  📝 Top-3 chunks after reranking:
    [1] The gate raises QualityGateFailed when the checkpoint does not succeed, and the Airflow DAG places the Gold ta...
    [2] The capstone quality gate is a Great Expectations checkpoint that validates the Silver table for unique line_i...
    [3] Silver runs a MERGE upsert keyed on line_id, the business key formed from InvoiceNo and StockCode, so price co...

  ⚠️  GROQ_API_KEY not set — returning the grounded context itself,
      already cited, instead of a generated answer.

  ✅ Answer:
    [Source 1] The gate raises QualityGateFailed when the checkpoint does not succeed, and the Airflow DAG places the Gold task downstream of it, so Gold never runs on data that failed validation.
[Source 2] The capstone quality gate is a Great Expectations checkpoint that validates the Silver table for unique line_id, non-null CustomerID, positive Quantity, UnitPrice and revenue, and a valid InvoiceNo pattern. The gate raises QualityGateFailed when the checkpo

4

## Deliverable 5 (cont.) — Lineage events

Every stage above emitted real OpenLineage events through the file transport.

In [ ]:
import json
from src.config import LINEAGE_LOG
import glob

for path in sorted(glob.glob(LINEAGE_LOG + "*")):
    print(f"--- {path} ---")
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            e = json.loads(line)
            print(f"  {e['eventType']:9s} {e['job']['namespace']}.{e['job']['name']:30s} {e['eventTime']}")


--- ./lineage_events/openlineage_run.log-20260722-052523.276239.json ---
  START     capstone.ingestion.produce              2026-07-22T05:25:23.275659+00:00
--- ./lineage_events/openlineage_run.log-20260722-052600.463688.json ---
  FAIL      capstone.ingestion.produce              2026-07-22T05:26:00.461706+00:00
--- ./lineage_events/openlineage_run.log-20260722-053756.119190.json ---
  START     capstone.ingestion.produce              2026-07-22T05:37:56.118735+00:00
--- ./lineage_events/openlineage_run.log-20260722-053802.706998.json ---
  COMPLETE  capstone.ingestion.produce              2026-07-22T05:38:02.706468+00:00
--- ./lineage_events/openlineage_run.log-20260722-053805.733480.json ---
  START     capstone.ingestion.consume_validate     2026-07-22T05:38:05.733110+00:00
--- ./lineage_events/openlineage_run.log-20260722-053841.724493.json ---
  COMPLETE  capstone.ingestion.consume_validate     2026-07-22T05:38:41.724130+00:00
--- ./lineage_events/openlineage_run.log-20260722-05

## Deliverable 4 — Airflow DAG

The DAG wires all of the above in one graph. It parses without a scheduler running, which
confirms the task dependencies are valid:

```
ingestion_produce -> ingestion_consume_validate -> lakehouse_bronze
  -> lakehouse_silver -> quality_gate -> [lakehouse_gold, rag_pipeline]
```

In [ ]:
import importlib.util, os, sys
import airflow

os.environ.setdefault("AIRFLOW_HOME", os.path.join(os.getcwd(), ".airflow"))
print("Airflow version:", airflow.__version__)


spec = importlib.util.spec_from_file_location(
    "capstone_pipeline_dag", "dags/capstone_pipeline_dag.py")
module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = module
spec.loader.exec_module(module)          # raises here if the DAG file is broken

dag = module.dag
print(f"\nDAG: {dag.dag_id} — {len(dag.tasks)} tasks\n")

edges    = {t.task_id: set(t.downstream_task_ids) for t in dag.tasks}
indegree = {tid: 0 for tid in edges}
for downs in edges.values():
    for d in downs:
        indegree[d] += 1

queue, order = sorted(t for t, d in indegree.items() if d == 0), []
while queue:
    tid = queue.pop(0)
    order.append(tid)
    for d in sorted(edges[tid]):
        indegree[d] -= 1
        if indegree[d] == 0:
            queue.append(d)
    queue.sort()

assert len(order) == len(edges), "Cycle detected in the DAG"

for tid in order:
    print(f"  {tid:30s} -> {sorted(edges[tid]) or '(end)'}")

gate_downstream = sorted(edges["quality_gate"])
print(f"\nGated by quality_gate: {gate_downstream}")
print("A failed gate leaves both of those skipped — the pipeline halts before Gold.")


Airflow version: 3.3.0
2026-07-22T06:15:14.113714Z [warning  ] The `airflow.operators.python.PythonOperator` attribute is deprecated. Please use `'airflow.providers.standard.operators.python.PythonOperator'`. [py.warnings] category=DeprecatedImportWarning filename=/content/capstone-modern-data-engineering/dags/capstone_pipeline_dag.py lineno=24

DAG: capstone_data_pipeline — 7 tasks

  ingestion_produce              -> ['ingestion_consume_validate']
  ingestion_consume_validate     -> ['lakehouse_bronze']
  lakehouse_bronze               -> ['lakehouse_silver']
  lakehouse_silver               -> ['quality_gate']
  quality_gate                   -> ['lakehouse_gold', 'rag_pipeline']
  lakehouse_gold                 -> (end)
  rag_pipeline                   -> (end)

Gated by quality_gate: ['lakehouse_gold', 'rag_pipeline']
A failed gate leaves both of those skipped — the pipeline halts before Gold.


## Proving the gate halts the pipeline

Tighten one expectation so Silver cannot pass, and watch the gate raise **and** emit a
`FAIL` lineage event. In the DAG, this is what leaves `lakehouse_gold` and `rag_pipeline`
skipped.

In [ ]:
import great_expectations as gx
import great_expectations.expectations as gxe
import pandas as pd

from src.config import SILVER_PATH
from src.lakehouse.spark_session import create_spark_session
from src.lineage.emitter import stage_lineage
from src.quality.expectations import QualityGateFailed

spark = create_spark_session("Capstone_GateFailureDemo")
silver_pdf = spark.read.format("delta").load(SILVER_PATH).toPandas()
spark.stop()

def impossible_gate(df):
    context = gx.get_context(mode="ephemeral")
    ds = context.data_sources.add_pandas("pandas_gate_demo")
    asset = ds.add_dataframe_asset(name="silver_demo")
    bd = asset.add_batch_definition_whole_dataframe("whole_df")
    suite = context.suites.add(gx.ExpectationSuite(name="impossible_suite"))
    # Every real line is under 1000 units, so this expectation must fail.
    suite.add_expectation(gxe.ExpectColumnValuesToBeBetween(column="Quantity", min_value=1000))
    vd = context.validation_definitions.add(
        gx.ValidationDefinition(name="impossible_validation", data=bd, suite=suite))
    cp = context.checkpoints.add(
        gx.Checkpoint(name="impossible_checkpoint", validation_definitions=[vd]))
    result = cp.run(batch_parameters={"dataframe": df})
    print(f"[GX] checkpoint success={result.success}")
    if not result.success:
        raise QualityGateFailed("Silver failed the tightened expectation on Quantity")

try:
    with stage_lineage("quality.gate_failure_demo"):
        impossible_gate(silver_pdf)
except QualityGateFailed as exc:
    print(f"\nPIPELINE HALTED: {exc}")
    print("A FAIL lineage event was emitted for capstone.quality.gate_failure_demo.")
    print("In the DAG, lakehouse_gold and rag_pipeline would now be skipped.")


2026-07-22T06:15:22.957158Z [info     ] OpenLineageClient will use `file` transport [openlineage.client.client] loc=client.py:136
[LINEAGE] START    | capstone.quality.gate_failure_demo | run 019f8876-ec4c-7122-843a-9f2924774736
2026-07-22T06:15:22.963407Z [info     ] Created temporary directory '/tmp/tmp10s7lsce' for ephemeral docs site [great_expectations.data_context.types.base] loc=base.py:1384
2026-07-22T06:15:22.967293Z [info     ] Loading 'datasources' ->
[]    [great_expectations.datasource.fluent.config] loc=config.py:191


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

[GX] checkpoint success=False
2026-07-22T06:15:23.170813Z [info     ] OpenLineageClient will use `file` transport [openlineage.client.client] loc=client.py:136
[LINEAGE] FAIL     | capstone.quality.gate_failure_demo | run 019f8876-ec4c-7122-843a-9f2924774736

PIPELINE HALTED: Silver failed the tightened expectation on Quantity
A FAIL lineage event was emitted for capstone.quality.gate_failure_demo.
In the DAG, lakehouse_gold and rag_pipeline would now be skipped.


## Run summary

| Deliverable | Evidence in this notebook |
| --- | --- |
| Ingestion | producer/consumer output, quarantine CSV preview, DLQ console consumer |
| Delta Lakehouse | Bronze append, MERGE metrics, Delta history, schema-enforcement rejection, Gold aggregate |
| RAG | vector + BM25 candidate counts, RRF fusion, reranked top-3, cited answer |
| Orchestration | parsed DAG with topological task order |
| Quality gate + lineage | GX checkpoint results, OpenLineage event log, forced FAIL demo |
